In [9]:
# ==============================
# 1. IMPORTS
# ==============================
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model


In [10]:
# ==============================
# 2. CONFIG
# ==============================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
data_path = "data.json"   # your dataset file


In [4]:
# ==============================
# 3. LOAD DATASET
# ==============================
dataset = load_dataset("json", data_files=data_path)["train"]

g:\v1\env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Map: 100%|██████████| 1651/1651 [00:00<00:00, 5551.00 examples/s]


In [13]:
# ==============================
# 4. FORMAT PROMPT
# ==============================
def format_example(example):
    replies = "\n".join(example["output"])

    text = (
        "### Instruction:\n"
        f"{example['instruction']}\n\n"
        "### Conversation:\n"
        f"{example['input']}\n\n"
        "### Response:\n"
        f"{replies}"
    )

    return {"text": text}
dataset = dataset.map(format_example)

Map: 100%|██████████| 1651/1651 [00:00<00:00, 8423.20 examples/s]


In [17]:
# ==============================
# 5. TOKENIZER
# ==============================
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


In [18]:
# ==============================
# 6. TOKENIZATION + LABELS (FIX)
# ==============================
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # ✅ IMPORTANT: add labels for loss
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

dataset = dataset.map(tokenize, batched=True)

# remove unused columns
dataset = dataset.remove_columns(["instruction", "input", "output", "text"])

Map: 100%|██████████| 1651/1651 [00:00<00:00, 6554.11 examples/s]


In [24]:

# ==============================
# 7. LOAD MODEL (4-bit QLoRA)
# ==============================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)
from peft import prepare_model_for_kbit_training

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# ✅ VERY IMPORTANT (FIXES YOUR ERROR)
model = prepare_model_for_kbit_training(model)

# reduce memory
model.gradient_checkpointing_enable()


  0%|          | 0/618 [07:02<?, ?it/s]


In [25]:
# ==============================
# 8. APPLY LoRA
# ==============================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

In [26]:
# ==============================
# 9. TRAINING CONFIG
# ==============================
training_args = TrainingArguments(
    output_dir="./model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)


In [27]:
# ==============================
# 10. TRAINER
# ==============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer
)


In [28]:
# ==============================
# 11. TRAIN 🚀
# ==============================
trainer.train()


                                                
  2%|▏         | 10/618 [00:18<18:25,  1.82s/it]

{'loss': 11.7901, 'grad_norm': 23.577314376831055, 'learning_rate': 0.00019741100323624597, 'epoch': 0.05}


                                                
  3%|▎         | 20/618 [00:36<17:49,  1.79s/it]

{'loss': 3.5278, 'grad_norm': 13.760140419006348, 'learning_rate': 0.0001941747572815534, 'epoch': 0.1}


                                                
  5%|▍         | 30/618 [00:54<17:56,  1.83s/it]

...

                                                 
 99%|█████████▊| 610/618 [17:24<00:12,  1.62s/it]

{'loss': 0.1679, 'grad_norm': 0.43728363513946533, 'learning_rate': 3.2362459546925567e-06, 'epoch': 2.95}


                                                 
100%|██████████| 618/618 [17:37<00:00,  1.71s/it]

{'train_runtime': 1057.4937, 'train_samples_per_second': 4.684, 'train_steps_per_second': 0.584, 'train_loss': 0.4449971704035515, 'epoch': 2.99}


TrainOutput(global_step=618, training_loss=0.4449971704035515, metrics={'train_runtime': 1057.4937, 'train_samples_per_second': 4.684, 'train_steps_per_second': 0.584, 'train_loss': 0.4449971704035515, 'epoch': 2.99})

In [29]:
# ==============================
# 12. SAVE MODEL
# ==============================
model.save_pretrained("smart_reply_model")
tokenizer.save_pretrained("smart_reply_model")

print("✅ Training Complete!")

✅ Training Complete!
